# Chapitre 11 - Cours 3 : nombre d'unités, weight decay et chiffres manuscrits

Ce notebook reproduit les deux exemples des sections 11.6 et 11.7 du livre :

1. les **données simulées** « somme de sigmoïdes » et « radiale » : erreur de test en fonction du
   nombre d'unités cachées et du paramètre de weight decay (figures 11.6, 11.7 et 11.8) ;
2. les **chiffres manuscrits** (codes postaux) : architectures du tableau 11.1, ajustement des
   réseaux entièrement connectés Net-1 et Net-2 sur un petit échantillon et courbes de
   performance au fil des époques (figure 11.11).

Il se termine par la discussion de la section 11.8 (forces et limites des réseaux).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch11_reseaux import MLP, n_weights, one_hot, sigmoid

## 1. Données simulées (section 11.6)

On simule deux modèles additifs $Y = f(X) + \varepsilon$ :

$$
\begin{aligned}
\text{somme de sigmoïdes : }\quad Y &= \sigma(a_1^T X) + \sigma(a_2^T X) + \varepsilon_1, \qquad p = 2,\ a_1 = (3,3)^T,\ a_2 = (3,-3)^T ;\\
\text{radial : }\quad Y &= \prod_{m=1}^{10} \phi(X_m) + \varepsilon_2, \qquad p = 10,\ \phi(t) = (2\pi)^{-1/2} e^{-t^2/2}.
\end{aligned}
$$

Les $X_j$ sont des gaussiennes centrées réduites indépendantes et les bruits sont gaussiens, de
variance choisie pour que le **rapport signal sur bruit**

$$\frac{\mathrm{Var}\big(\mathbb E(Y\mid X)\big)}{\mathrm{Var}\big(Y - \mathbb E(Y \mid X)\big)} = \frac{\mathrm{Var}(f(X))}{\mathrm{Var}(\varepsilon)} \qquad \text{(éq. (11.18))}$$

vaille 4 dans les deux cas. On tire $N = 100$ observations d'apprentissage et $10\,000$ de test.

Pour le modèle radial, la variance de $f$ se calcule exactement. Par indépendance des $X_m$,
$\mathbb E f(X) = \prod_m \mathbb E\phi(X_m) = \big(\mathbb E\phi(X_1)\big)^{10}$ et
$\mathbb E f(X)^2 = \big(\mathbb E\phi(X_1)^2\big)^{10}$. Or, pour $X_1 \sim \mathcal N(0,1)$,

$$\mathbb E\,\phi(X_1) = \int \phi(t)^2\,dt = \frac{1}{2\pi}\int e^{-t^2} dt = \frac{\sqrt\pi}{2\pi} = \frac{1}{2\sqrt\pi},
\qquad \mathbb E\,\phi(X_1)^2 = \int \phi(t)^3 dt = \frac{1}{(2\pi)^{3/2}} \sqrt{\frac{2\pi}{3}} = \frac{1}{2\pi\sqrt 3},$$

(en utilisant $\int e^{-ct^2}dt = \sqrt{\pi/c}$), d'où
$\mathrm{Var} f(X) = (2\pi\sqrt3)^{-10} - (2\sqrt\pi)^{-20}$.

In [2]:
a1, a2 = np.array([3.0, 3.0]), np.array([3.0, -3.0])

def f_sigmoides(X):
    return sigmoid(X @ a1) + sigmoid(X @ a2)

def f_radiale(X):
    return np.prod(np.exp(-X ** 2 / 2) / np.sqrt(2 * np.pi), axis=1)

var_rad = (2 * np.pi * np.sqrt(3)) ** -10 - (2 * np.sqrt(np.pi)) ** -20
rng = np.random.RandomState(0)
print("Var f radiale : exacte = %.4e, Monte-Carlo = %.4e"
      % (var_rad, np.var(f_radiale(rng.randn(1_000_000, 10)))))

Var f radiale : exacte = 3.2730e-11, Monte-Carlo = 3.2572e-11


In [3]:
def simuler(f, p, var_f, rng, N=100, N_test=10_000):
    sigma2 = var_f / 4                                   # RSB = 4
    X, Xt = rng.randn(N, p), rng.randn(N_test, p)
    y = f(X) + np.sqrt(sigma2) * rng.randn(N)
    yt = f(Xt) + np.sqrt(sigma2) * rng.randn(N_test)
    return X, y, Xt, yt, sigma2

rng = np.random.RandomState(1)
var_sig = np.var(f_sigmoides(rng.randn(1_000_000, 2)))
sim = {"Somme de sigmoïdes": simuler(f_sigmoides, 2, var_sig, rng),
       "Radial": simuler(f_radiale, 10, var_rad, rng)}

On ajuste, sur l'**unique** échantillon d'apprentissage, des réseaux à $M = 0, 1, \dots, 10$ unités
cachées, pour **10 tirages** des poids initiaux, et l'on rapporte l'erreur de test
$\mathbb E_{\text{test}}(Y - \hat f(X))^2$ **divisée par l'erreur de Bayes** $\sigma^2 = \mathrm{Var}(\varepsilon)$
(rappel : en régression quadratique, l'erreur de Bayes est la variance du bruit, atteinte par
$f$ elle-même). Le modèle à $M = 0$ unité est la régression linéaire par moindres carrés.

Détails de mise en œuvre :

* optimisation par L-BFGS (au plus 1000 itérations), poids initiaux uniformes sur $[-0{,}7 ; 0{,}7]$ ;
* la fonction radiale prend des valeurs minuscules (de l'ordre de $10^{-5}$) : on **standardise**
  la réponse avant l'ajustement (puis on revient à l'échelle d'origine), faute de quoi le
  weight decay n'aurait aucun sens.

In [4]:
def erreurs_relatives(nom, M, lam, n_starts=10):
    X, y, Xt, yt, sigma2 = sim[nom]
    mu, sd = (y.mean(), y.std()) if nom == "Radial" else (0.0, 1.0)
    out = []
    for seed in range(n_starts):
        net = MLP(n_hidden=M, weight_decay=lam, random_state=seed)
        net.fit(X, (y - mu) / sd, max_iter=1000)
        pred = mu + sd * net.predict(Xt)[:, 0]
        out.append(np.mean((yt - pred) ** 2) / sigma2)
        if M == 0:                 # modèle linéaire : un seul départ suffit (critère convexe)
            out = out * n_starts
            break
    return out

Certains départs donnent des erreurs énormes (le réseau extrapole très mal loin des données) ;
comme dans le livre, on **tronque** l'axe vertical, et l'on indique au-dessus de chaque boîte le
nombre de valeurs hors cadre.